# LeetCode #1199: Minimum Time to Build Blocks

https://leetcode.com/problems/minimum-time-to-build-blocks/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n! \cdot 2^n)$ | $O(n)$ |
| **Optimal: Huffman Greedy (Min-Heap) ★** | $O(n \log n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Try every possible order of splitting workers and assigning blocks, simulating parallel execution. The combinatorial explosion of worker/block assignments makes this completely infeasible for large inputs.

### Optimal: Huffman Greedy (Min-Heap) ★
The key insight mirrors Huffman coding: to minimise the bottleneck, always merge the two cheapest remaining tasks. Push all block times onto a min-heap. Repeatedly pop the two smallest, push `max(a, b) + split` (workers build in parallel; the split cost is paid before they diverge). The last value in the heap is the answer.

**Why this is better than Brute Force:** Instead of exploring exponential orderings it greedily collapses the problem in $O(n \log n)$, exploiting the parallel-execution structure.

**Constraints:**
* $1 \leq$ `blocks.length` $\leq 1000$
* $1 \leq$ `blocks[i]` $\leq 10^5$
* $1 \leq$ `split` $\leq 100$

## Solutions

### C#

In [ ]:
using System.Collections.Generic;

public class Solution {
    public int MinBuildTime(int[] blocks, int split) {
        // Load all building times into a min-heap
        var heap = new SortedList<int, int>();
        void Push(int v) { heap.TryGetValue(v, out int c); heap[v] = c + 1; }
        int Pop() { var kv = heap.Min; if (kv.Value == 1) heap.Remove(kv.Key); else heap[kv.Key]--; return kv.Key; }

        foreach (var b in blocks) Push(b);

        // Huffman merge: always collapse the two cheapest tasks
        while (heap.Count > 1) {
            int a = Pop(), b = Pop();
            // Workers run in parallel, so only the slower one determines the path length
            Push(Math.Max(a, b) + split);
        }
        return heap.Min.Key;
    }
}

### Python

In [ ]:
import heapq
from typing import List

class Solution:
    def min_build_time(self, blocks: List[int], split: int) -> int:
        # Load all building times into a min-heap
        heap = blocks[:]
        heapq.heapify(heap)

        # Huffman merge: always collapse the two cheapest tasks
        while len(heap) > 1:
            a = heapq.heappop(heap)
            b = heapq.heappop(heap)
            # Workers run in parallel, so only the slower one determines the path length
            heapq.heappush(heap, max(a, b) + split)

        return heap[0]

### Go

In [ ]:
import "container/heap"

type IntHeap []int
func (h IntHeap) Len() int           { return len(h) }
func (h IntHeap) Less(i, j int) bool { return h[i] < h[j] }
func (h IntHeap) Swap(i, j int)      { h[i], h[j] = h[j], h[i] }
func (h *IntHeap) Push(x any)        { *h = append(*h, x.(int)) }
func (h *IntHeap) Pop() any          { old := *h; n := len(old); x := old[n-1]; *h = old[:n-1]; return x }

func minBuildTime(blocks []int, split int) int {
    h := IntHeap(append([]int{}, blocks...))
    heap.Init(&h)

    // Huffman merge: always collapse the two cheapest tasks
    for h.Len() > 1 {
        a := heap.Pop(&h).(int)
        b := heap.Pop(&h).(int)
        // Workers run in parallel, so only the slower one determines the path length
        merged := b + split // b >= a after two pops from min-heap
        if a > b { merged = a + split }
        heap.Push(&h, merged)
    }
    return h[0]
}

### Rust

In [ ]:
use std::collections::BinaryHeap;
use std::cmp::Reverse;

impl Solution {
    pub fn min_build_time(blocks: Vec<i32>, split: i32) -> i32 {
        // Load all building times into a min-heap (Reverse for min-heap behaviour)
        let mut heap: BinaryHeap<Reverse<i32>> = blocks.iter().map(|&b| Reverse(b)).collect();

        // Huffman merge: always collapse the two cheapest tasks
        while heap.len() > 1 {
            let Reverse(a) = heap.pop().unwrap();
            let Reverse(b) = heap.pop().unwrap();
            // Workers run in parallel, so only the slower one determines the path length
            heap.push(Reverse(a.max(b) + split));
        }
        heap.pop().unwrap().0
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `blocks = [1, 2], split = 5`
One worker splits (cost 5), then each builds in parallel: max(1, 2) = 2. Total time = $5 + 2 = 7$. The heap merges [1, 2] → [7].

### 2. Slightly Complex
**Input:** `blocks = [1, 2, 3], split = 1`
Merge 1 and 2 → max(1,2)+1=3, heap=[3,3]. Merge 3 and 3 → max(3,3)+1=4. Answer: **4**.

### 3. Edge Case: Time Factor
**Input:** `blocks = [1]*999 + [100000], split = 1`
The Huffman tree has $n-1$ internal merge nodes; all $O(n \log n)$ heap operations run. The dominant path through the deepest leaf drives the result.

### 4. Edge Case: Space Factor
**Input:** `blocks = [100000]*1000, split = 100`
All blocks are equal. The heap performs $\approx n-1$ merges, each pushing one new element, so the heap size oscillates around $n/2$ in worst case — $O(n)$ space.

### 5. Almost-Impossible but Plausible
**Input:** `blocks = [1, 99999], split = 99999`
Merging gives max(1, 99999)+99999 = 199998. But skipping the split (single worker builds both sequentially) costs 1+99999=100000 — except the problem guarantees a single worker can only build one block. So the answer is 199998, counterintuitively large when split cost dominates.